In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torch.optim as optim
from tqdm import tqdm
import os
import numpy as np
import time
import matplotlib.pyplot as plt
import math
import gc
import imageio.v3 as io

## Architecture and DataLoader

In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

In [3]:
class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        # Encoder
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = DoubleConv(64, 128)
        self.down2 = DoubleConv(128, 256)
        self.down3 = DoubleConv(256, 512)
        
        # Decoder
        self.up1 = DoubleConv(512 + 256, 256)
        self.up2 = DoubleConv(256 + 128, 128)
        self.up3 = DoubleConv(128 + 64, 64)
        
        # Convolution
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
        
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(self.pool(x1))
        x3 = self.down2(self.pool(x2))
        x4 = self.down3(self.pool(x3))

        # Decoder
        x = self.upsample(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.up1(x)
        
        x = self.upsample(x)
        x = torch.cat([x, x2], dim=1)
        x = self.up2(x)
        
        x = self.upsample(x)
        x = torch.cat([x, x1], dim=1)
        x = self.up3(x)
        
        # Final output
        logits = self.outc(x)
        return logits

In [4]:
class PhaseRetrievalDataset(Dataset):
    def __init__(self, data_path, data_type='plane_wave'):
        self.labels = np.load(os.path.join(data_path, 'labels.npy'))
        
        if data_type == 'plane_wave':
            self.inputs = np.load(os.path.join(data_path, 'inputs_plane_wave.npy'))
        elif data_type == 'vortex':
            self.inputs = np.load(os.path.join(data_path, 'inputs_vortex.npy'))
        else:
            raise ValueError("data_type must be 'plane_wave' or 'vortex'")
        
        print(f"Loaded dataset from {data_path} with {len(self.labels)} samples.")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = self.labels[idx]
        input_data = self.inputs[idx]
        
        label = torch.from_numpy(label).unsqueeze(0).float()
        input_data = torch.from_numpy(input_data).unsqueeze(0).float()
        
        return input_data, label

In [5]:
#New
class PhaseRetrievalDatasetLazy(Dataset):
    def __init__(self, data_path, data_type='plane_wave'):
        self.labels_path = os.path.join(data_path, 'labels.npy')
        self.inputs_path = os.path.join(data_path, f'inputs_{data_type}.npy')
        
        # Open the memory-mapped files ONCE and store the objects
        self.labels = np.load(self.labels_path, mmap_mode='r')
        self.inputs = np.load(self.inputs_path, mmap_mode='r')
        
        # Get the length from the shape of the memmap object
        self.length = self.labels.shape[0]
        
    def __len__(self):
        return self.length
        
    def __getitem__(self, idx):
        label = self.labels[idx]
        input_data = self.inputs[idx]
        label_tensor = torch.from_numpy(label.copy()).unsqueeze(0).float()
        input_tensor = torch.from_numpy(input_data.copy()).unsqueeze(0).float()
        
        return input_tensor, label_tensor

## Hyperparameters

In [6]:
NUM_TRAIN = 400
NUM_VAL = 100
NUM_TEST = 100
NUM_EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
N = 256
# Define paths for saving results
RESULTS_PATH = os.path.join('..', 'results')
MODEL_SAVE_PATH = os.path.join(RESULTS_PATH, 'models')
PLOT_SAVE_PATH = os.path.join(RESULTS_PATH, 'plots')
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(PLOT_SAVE_PATH, exist_ok=True)
# Paths for loading data
PROCESSED_DATA_PATH = os.path.join('..', 'data', 'processed', 'celeba_128x128')
TRAIN_PATH = os.path.join(PROCESSED_DATA_PATH, 'train')
VAL_PATH = os.path.join(PROCESSED_DATA_PATH, 'val')
TEST_PATH = os.path.join(PROCESSED_DATA_PATH, 'test')
device = "cpu"
print(f"Using device: {device}")

Using device: cpu


## Training

In [7]:
def train_model(model, model_type, train_loader, val_loader, num_epochs, save_path):
    print(f"\n--- Starting Training for {model_type} model ---")
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        print(f"Epoch {epoch+1} -> Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
        
    torch.save(model.state_dict(), save_path)
    print(f"--- Model saved to {save_path} ---")
    return model, history

In [9]:
# TRAIN THE PLANE WAVE MODEL
train_dataset_plane = Subset(PhaseRetrievalDataset(TRAIN_PATH, 'plane_wave'), range(NUM_TRAIN))
val_dataset_plane = Subset(PhaseRetrievalDataset(VAL_PATH, 'plane_wave'), range(NUM_VAL))
train_loader_plane = DataLoader(train_dataset_plane, BATCH_SIZE, shuffle=True)
val_loader_plane = DataLoader(val_dataset_plane, BATCH_SIZE, shuffle=False)
model_plane = UNet(1, 1).to(device)
save_path_plane = os.path.join(MODEL_SAVE_PATH, 'unet_plane_wave.pth')
trained_model_plane, history_plane = train_model(model_plane, "plane_wave", train_loader_plane, val_loader_plane, NUM_EPOCHS, save_path_plane)

Loaded dataset from ..\data\processed\celeba_128x128\train with 5000 samples.
Loaded dataset from ..\data\processed\celeba_128x128\val with 1000 samples.

--- Starting Training for plane_wave model ---


Epoch 1/3 [Training]: 100%|███████████████████████████████████████████████| 25/25 [16:51<00:00, 40.45s/it, loss=0.0855]


Epoch 1 -> Train Loss: 0.101779, Val Loss: 0.107249


Epoch 2/3 [Training]: 100%|███████████████████████████████████████████████| 25/25 [15:16<00:00, 36.67s/it, loss=0.0762]


Epoch 2 -> Train Loss: 0.078039, Val Loss: 0.073791


Epoch 3/3 [Training]: 100%|███████████████████████████████████████████████| 25/25 [15:18<00:00, 36.74s/it, loss=0.0642]


Epoch 3 -> Train Loss: 0.070486, Val Loss: 0.209183
--- Model saved to ..\results\models\unet_plane_wave.pth ---


In [10]:
# TRAIN THE VORTEX MODEL
train_dataset_vortex = Subset(PhaseRetrievalDataset(TRAIN_PATH, 'vortex'), range(NUM_TRAIN))
val_dataset_vortex = Subset(PhaseRetrievalDataset(VAL_PATH, 'vortex'), range(NUM_VAL))
train_loader_vortex = DataLoader(train_dataset_vortex, BATCH_SIZE, shuffle=True)
val_loader_vortex = DataLoader(val_dataset_vortex, BATCH_SIZE, shuffle=False)
model_vortex = UNet(1, 1).to(device)
save_path_vortex = os.path.join(MODEL_SAVE_PATH, 'unet_vortex.pth')
trained_model_vortex, history_vortex = train_model(model_vortex, "vortex", train_loader_vortex, val_loader_vortex, NUM_EPOCHS, save_path_vortex)

Loaded dataset from ..\data\processed\celeba_128x128\train with 5000 samples.
Loaded dataset from ..\data\processed\celeba_128x128\val with 1000 samples.

--- Starting Training for vortex model ---


Epoch 1/3 [Training]: 100%|███████████████████████████████████████████████| 25/25 [15:05<00:00, 36.23s/it, loss=0.0829]


Epoch 1 -> Train Loss: 0.097899, Val Loss: 0.102814


Epoch 2/3 [Training]: 100%|███████████████████████████████████████████████| 25/25 [15:07<00:00, 36.29s/it, loss=0.0753]


Epoch 2 -> Train Loss: 0.075184, Val Loss: 0.072639


Epoch 3/3 [Training]: 100%|███████████████████████████████████████████████| 25/25 [15:03<00:00, 36.13s/it, loss=0.0696]


Epoch 3 -> Train Loss: 0.068313, Val Loss: 0.065341
--- Model saved to ..\results\models\unet_vortex.pth ---


In [7]:
def load_models(device):
    print("Loading pre-trained models")
    model_path_plane = os.path.join(MODEL_SAVE_PATH, 'unet_plane_wave.pth')
    model_path_vortex = os.path.join(MODEL_SAVE_PATH, 'unet_vortex.pth')
    
    model_plane = UNet(1, 1).to(device)
    model_vortex = UNet(1, 1).to(device)
    
    model_plane.load_state_dict(torch.load(model_path_plane, map_location=torch.device(device)))
    model_vortex.load_state_dict(torch.load(model_path_vortex, map_location=torch.device(device)))
    
    model_plane.eval()
    model_vortex.eval()
    
    print("Models loaded successfully")
    return model_plane, model_vortex

def psnr(label, output, max_val=1.0):
    mse = torch.mean((label - output) ** 2)
    return 20 * math.log10(max_val / math.sqrt(mse)) if mse > 0 else float('inf')

In [9]:
def evaluate_models(model_plane, model_vortex, test_loader_plane, test_loader_vortex, device):
    print("Starting evaluation")
    total_mae_plane, total_psnr_plane = 0.0, 0.0
    total_mae_vortex, total_psnr_vortex = 0.0, 0.0
    num_samples = 0
    
    model_plane.eval()
    model_vortex.eval()
    
    with torch.no_grad():
        for batch_idx, ((inputs_plane, labels_plane), (inputs_vortex, labels_vortex)) in enumerate(
            zip(test_loader_plane, test_loader_vortex)):
            
            # Move to device
            inputs_plane = inputs_plane.to(device)
            inputs_vortex = inputs_vortex.to(device)
            labels = labels_plane.to(device)
            
            # Get predictions
            preds_plane = model_plane(inputs_plane)
            preds_vortex = model_vortex(inputs_vortex)
            
            # Calculate metrics
            batch_size = labels.size(0)
            mae_plane = nn.L1Loss()(preds_plane, labels).item()
            mae_vortex = nn.L1Loss()(preds_vortex, labels).item()
            
            # Calculate PSNR for each sample in the batch
            for i in range(batch_size):
                psnr_plane = psnr(labels[i:i+1], preds_plane[i:i+1])
                psnr_vortex = psnr(labels[i:i+1], preds_vortex[i:i+1])
                total_psnr_plane += psnr_plane
                total_psnr_vortex += psnr_vortex
            
            total_mae_plane += mae_plane * batch_size
            total_mae_vortex += mae_vortex * batch_size
            num_samples += batch_size
            
            # Clean up
            del inputs_plane, inputs_vortex, labels, preds_plane, preds_vortex
            
            if batch_idx % 5 == 0:
                print(f"Processed {num_samples}/{NUM_TEST} samples")
                gc.collect()
    
    # Calculate averages
    avg_mae_plane = total_mae_plane / num_samples
    avg_psnr_plane = total_psnr_plane / num_samples
    avg_mae_vortex = total_mae_vortex / num_samples
    avg_psnr_vortex = total_psnr_vortex / num_samples
    
    print("\nQuantitative Results on Test Set")
    print(f"Plane Wave Model -> MAE: {avg_mae_plane:.6f} | PSNR: {avg_psnr_plane:.2f} dB")
    print(f"Vortex Model     -> MAE: {avg_mae_vortex:.6f} | PSNR: {avg_psnr_vortex:.2f} dB")
    
    return {
        'mae_plane': avg_mae_plane,
        'psnr_plane': avg_psnr_plane,
        'mae_vortex': avg_mae_vortex,
        'psnr_vortex': avg_psnr_vortex
    }

In [11]:
print("Loading pre-trained models")
model_path_plane = os.path.join(MODEL_SAVE_PATH, 'unet_plane_wave.pth')
model_path_vortex = os.path.join(MODEL_SAVE_PATH, 'unet_vortex.pth')
model_plane = UNet(1, 1).to(device)
model_vortex = UNet(1, 1).to(device)
model_plane.load_state_dict(torch.load(model_path_plane, map_location=torch.device(device)))
model_vortex.load_state_dict(torch.load(model_path_vortex, map_location=torch.device(device)))
model_plane.eval()
model_vortex.eval()
print("Models loaded successfully")

def to_uint8(float_array):
    # Clip the array to the expected [0, 1] range first
    clipped = np.clip(float_array, 0, 1)
    # Scale to [0, 255]
    scaled = clipped * 255.0
    # Convert to unsigned 8-bit integer
    return scaled.astype(np.uint8)

# Get the test datasets
test_dataset_plane = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'plane_wave'), range(5))
test_dataset_vortex = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'vortex'), range(5))

print(f"\nGenerating and saving {5} sets of prediction images...")
for i in range(5):
    print(f"  Processing sample {i+1}/{5}...")
    
    # Get one sample at a time to keep memory usage minimal
    inputs_plane, label = test_dataset_plane[i]
    inputs_vortex, _ = test_dataset_vortex[i]
    
    # Add batch dimension for the model
    inputs_plane = inputs_plane.unsqueeze(0)
    inputs_vortex = inputs_vortex.unsqueeze(0)
    
    # Generate predictions
    with torch.no_grad():
        pred_plane = model_plane(inputs_plane.to(device)).cpu()
        pred_vortex_raw = model_vortex(inputs_vortex.to(device)).cpu()

    # Perform Vortex Phase Correction
    x0 = np.linspace(-(N - 1) / 2.0, (N - 1) / 2.0, N, endpoint=True)
    x_grid, y_grid = np.meshgrid(x0, x0)
    vortex_phase = np.arctan2(y_grid, x_grid) / np.pi
    vortex_phase_tensor = torch.from_numpy(vortex_phase).unsqueeze(0).unsqueeze(0).float()
    pred_vortex_corrected = pred_vortex_raw - vortex_phase_tensor
    
    # Convert all tensors to saveable numpy arrays
    label_np = label.squeeze().numpy()
    pred_plane_np = pred_plane.squeeze().numpy()
    pred_vortex_raw_np = pred_vortex_raw.squeeze().numpy()
    pred_vortex_corrected_np = pred_vortex_corrected.squeeze().numpy()
    
    # --- Save each image individually using the safe imageio writer ---
    io.imwrite(os.path.join(PLOT_SAVE_PATH, f'sample_{i}_1_ground_truth.png'), to_uint8(label_np))
    io.imwrite(os.path.join(PLOT_SAVE_PATH, f'sample_{i}_2_pred_plane.png'), to_uint8(pred_plane_np))
    io.imwrite(os.path.join(PLOT_SAVE_PATH, f'sample_{i}_3_pred_vortex_raw.png'), to_uint8(pred_vortex_raw_np))
    io.imwrite(os.path.join(PLOT_SAVE_PATH, f'sample_{i}_4_pred_vortex_corrected.png'), to_uint8(pred_vortex_corrected_np))

    # Clean up memory after each loop
    del inputs_plane, label, inputs_vortex, pred_plane, pred_vortex_raw, pred_vortex_corrected
    gc.collect()

print("\n--- SCRIPT COMPLETE ---")
print(f"All prediction images have been saved to the folder: {PLOT_SAVE_PATH}")
print("You can now assemble these images in your presentation slides.")

Loading pre-trained models...
Models loaded successfully.

Generating and saving 5 sets of prediction images...
  Processing sample 1/5...
  Processing sample 2/5...
  Processing sample 3/5...
  Processing sample 4/5...
  Processing sample 5/5...

--- SCRIPT COMPLETE ---
All prediction images have been saved to the folder: ..\results\plots
You can now assemble these images in your presentation slides.


In [ ]:
NUM_IMAGES_TO_PLOT = 4
NUM_TEST = 100
test_dataset_plane = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'plane_wave'), range(NUM_TEST))
test_loader_plane = DataLoader(test_dataset_plane, BATCH_SIZE, shuffle=False)
test_dataset_vortex = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'vortex'), range(NUM_TEST))
test_loader_vortex = DataLoader(test_dataset_vortex, BATCH_SIZE, shuffle=False)
total_mae_plane, total_psnr_plane, total_mae_vortex, total_psnr_vortex = 0, 0, 0, 0
hist_errors_plane, hist_errors_vortex = [], []
model_plane, model_vortex = load_models(device)
with torch.no_grad():
    for (inputs_plane, labels), (inputs_vortex, _) in tqdm(zip(test_loader_plane, test_loader_vortex), total=len(test_loader_plane), desc="Evaluating Test Set"):
        inputs_plane, inputs_vortex, labels = inputs_plane.to(device), inputs_vortex.to(device), labels.to(device)
        preds_plane = model_plane(inputs_plane)
        total_mae_plane += nn.L1Loss()(preds_plane, labels).item() * labels.size(0)
        total_psnr_plane += psnr(labels, preds_plane) * labels.size(0)
        hist_errors_plane.append(torch.abs(preds_plane - labels).cpu().numpy().flatten())
        preds_vortex = model_vortex(inputs_vortex)
        total_mae_vortex += nn.L1Loss()(preds_vortex, labels).item() * labels.size(0)
        total_psnr_vortex += psnr(labels, preds_vortex) * labels.size(0)
        hist_errors_vortex.append(torch.abs(preds_vortex - labels).cpu().numpy().flatten())
        del inputs_plane, inputs_vortex, labels, preds_plane, preds_vortex
        gc.collect()
avg_mae_plane = total_mae_plane / NUM_TEST; avg_psnr_plane = total_psnr_plane / NUM_TEST
avg_mae_vortex = total_mae_vortex / NUM_TEST; avg_psnr_vortex = total_psnr_vortex / NUM_TEST
error_plane_full = np.concatenate(hist_errors_plane); error_vortex_full = np.concatenate(hist_errors_vortex)
print("\nQuantitative Results on Test Set")
print(f"Plane Wave Model -> MAE: {avg_mae_plane:.6f} | PSNR: {avg_psnr_plane:.2f} dB")
print(f"Vortex Model     -> MAE: {avg_mae_vortex:.6f} | PSNR: {avg_psnr_vortex:.2f} dB")
del hist_errors_plane, hist_errors_vortex
gc.collect()

print("\nGenerating and saving presentation plots")
def to_uint8(float_array):
    clipped = np.clip(float_array, 0, 1)
    scaled = clipped * 255.0
    return scaled.astype(np.uint8)

def gray_to_rgb(gray_image):
    return np.stack((gray_image,) * 3, axis=-1)

plot_dataset_plane = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'plane_wave'), range(NUM_IMAGES_TO_PLOT))
plot_loader_plane = DataLoader(plot_dataset_plane, batch_size=NUM_IMAGES_TO_PLOT)
plot_dataset_vortex = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'vortex'), range(NUM_IMAGES_TO_PLOT))
plot_loader_vortex = DataLoader(plot_dataset_vortex, batch_size=NUM_IMAGES_TO_PLOT)
inputs_plane, labels = next(iter(plot_loader_plane))
inputs_vortex, _ = next(iter(plot_loader_vortex))

# Get predictions for the plot batch
with torch.no_grad():
    preds_plane = model_plane(inputs_plane.to(device)).cpu()
    preds_vortex_raw = model_vortex(inputs_vortex.to(device)).cpu()

# Perform Vortex Correction
x0 = np.linspace(-(N - 1) / 2.0, (N - 1) / 2.0, N, endpoint=True)
x_grid, y_grid = np.meshgrid(x0, x0)
vortex_phase = np.arctan2(y_grid, x_grid) / np.pi
vortex_phase_tensor = torch.from_numpy(vortex_phase).unsqueeze(0).unsqueeze(0).float()
preds_vortex_corrected = preds_vortex_raw - vortex_phase_tensor

# Side-by-Side Comparison Images
for i in range(NUM_IMAGES_TO_PLOT):
    img_gt = labels[i, 0].numpy()
    img_plane = preds_plane[i, 0].numpy()
    img_vortex_raw = preds_vortex_raw[i, 0].numpy()
    img_vortex_corr = preds_vortex_corrected[i, 0].numpy()
    
    # Convert all to saveable uint8 format
    images_to_stack = [to_uint8(img) for img in [img_gt, img_plane, img_vortex_raw, img_vortex_corr]]
    
    # Add a border between images for clarity
    border = np.ones((images_to_stack[0].shape[0], 10), dtype=np.uint8) * 255
    composite_image = np.hstack([images_to_stack[0], border, images_to_stack[1], border, images_to_stack[2], border, images_to_stack[3]])
    
    # Save the composite image
    save_path = os.path.join(PLOT_SAVE_PATH, f'comparison_sample_{i+1}.png')
    io.imwrite(save_path, composite_image)
print(f"Saved {NUM_IMAGES_TO_PLOT} side-by-side comparison images to {PLOT_SAVE_PATH}")


# Error Maps
error_plane = torch.abs(preds_plane - labels); error_vortex = torch.abs(preds_vortex_raw - labels)
cmap = plt.get_cmap('inferno')
for i in range(NUM_IMAGES_TO_PLOT):
    norm_error_plane = error_plane[i, 0].numpy() / 0.5; norm_error_vortex = error_vortex[i, 0].numpy() / 0.5
    error_plane_rgb = (cmap(np.clip(norm_error_plane, 0, 1))[:,:,:3] * 255).astype(np.uint8)
    error_vortex_rgb = (cmap(np.clip(norm_error_vortex, 0, 1))[:,:,:3] * 255).astype(np.uint8)
    img_gt_uint8 = to_uint8(labels[i, 0].numpy())
    img_gt_rgb = gray_to_rgb(img_gt_uint8)
    border_rgb = gray_to_rgb(np.ones((img_gt_uint8.shape[0], 10), dtype=np.uint8) * 255)
    composite_error_image = np.hstack([img_gt_rgb, border_rgb, error_plane_rgb, border_rgb, error_vortex_rgb])
    save_path = os.path.join(PLOT_SAVE_PATH, f'error_map_sample_{i+1}.png')
    io.imwrite(save_path, composite_error_image)
print(f"Saved {NUM_IMAGES_TO_PLOT} error map comparison images to {PLOT_SAVE_PATH}")


# Error Histogram
plt.figure(figsize=(10, 6))
plt.hist(error_plane_full, bins=100, range=(0, 0.5), density=True, alpha=0.7, label='Plane Wave Error')
plt.hist(error_vortex_full, bins=100, range=(0, 0.5), density=True, alpha=0.7, label='Vortex Error')
plt.title('Distribution of Absolute Pixel Errors on Test Set', fontsize=16)
plt.xlabel('Absolute Error'); plt.ylabel('Probability Density'); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(PLOT_SAVE_PATH, 'error_histograms.png'))
plt.show()
plt.close()
print(f"Saved error histogram plot to {PLOT_SAVE_PATH}")

# Final cleanup
gc.collect()
print("\nAll plots and metrics have been generated and saved successfully!")

Loading pre-trained models
Models loaded successfully


Evaluating Test Set: 100%|███████████████████████████████████████████████████████████████| 7/7 [02:26<00:00, 20.95s/it]



Quantitative Results on Test Set
Plane Wave Model -> MAE: 0.206309 | PSNR: 12.02 dB
Vortex Model     -> MAE: 0.066045 | PSNR: 16.00 dB

Generating and saving presentation plots
Saved 4 side-by-side comparison images to ..\results\plots


In [8]:
NUM_IMAGES_TO_PLOT = 4
BATCH_SIZE = 4
test_dataset_plane = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'plane_wave'), range(NUM_IMAGES_TO_PLOT))
test_loader_plane = DataLoader(test_dataset_plane, batch_size=BATCH_SIZE, shuffle=False)
test_dataset_vortex = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'vortex'), range(NUM_IMAGES_TO_PLOT))
test_loader_vortex = DataLoader(test_dataset_vortex, batch_size=BATCH_SIZE, shuffle=False)
inputs_plane, labels = next(iter(test_loader_plane))
inputs_vortex, _ = next(iter(test_loader_vortex))
model_plane, model_vortex = load_models(device)
with torch.no_grad():
    preds_plane = model_plane(inputs_plane.to(device)).cpu()
    preds_vortex_raw = model_vortex(inputs_vortex.to(device)).cpu()

error_plane = torch.abs(preds_plane - labels)
error_vortex = torch.abs(preds_vortex_raw - labels)
cmap = plt.get_cmap('inferno')

def to_uint8(float_array):
    clipped = np.clip(float_array, 0, 1); scaled = clipped * 255.0; return scaled.astype(np.uint8)

def gray_to_rgb(gray_image):
    return np.stack((gray_image,) * 3, axis=-1)

for i in range(NUM_IMAGES_TO_PLOT):
    # Normalize error maps to [0, 1] for colormapping
    norm_error_plane = error_plane[i, 0].numpy() / 0.5 # Normalize by a max error of 0.5
    norm_error_vortex = error_vortex[i, 0].numpy() / 0.5
    
    # Apply colormap and convert to uint8
    error_plane_rgb = (cmap(np.clip(norm_error_plane, 0, 1))[:,:,:3] * 255).astype(np.uint8)
    error_vortex_rgb = (cmap(np.clip(norm_error_vortex, 0, 1))[:,:,:3] * 255).astype(np.uint8)
    
    # Convert ground truth to 3D RGB format
    img_gt_uint8 = to_uint8(labels[i, 0].numpy())
    img_gt_rgb = gray_to_rgb(img_gt_uint8)
    border_rgb = gray_to_rgb(np.ones((img_gt_uint8.shape[0], 10), dtype=np.uint8) * 255)

    # Stack and save
    composite_error_image = np.hstack([img_gt_rgb, border_rgb, error_plane_rgb, border_rgb, error_vortex_rgb])
    save_path = os.path.join(PLOT_SAVE_PATH, f'error_map_sample_{i+1}.png')
    io.imwrite(save_path, composite_error_image)

del inputs_plane, labels, inputs_vortex, preds_plane, preds_vortex_raw, error_plane, error_vortex
gc.collect()

print(f"Saved {NUM_IMAGES_TO_PLOT} error map comparison images to {PLOT_SAVE_PATH}")

Loading pre-trained models
Models loaded successfully
Saved 4 error map comparison images to ..\results\plots


In [12]:
test_dataset_plane = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'plane_wave'), range(NUM_TEST))
test_loader_plane = DataLoader(test_dataset_plane, BATCH_SIZE, shuffle=False)
test_dataset_vortex = Subset(PhaseRetrievalDatasetLazy(TEST_PATH, 'vortex'), range(NUM_TEST))
test_loader_vortex = DataLoader(test_dataset_vortex, BATCH_SIZE, shuffle=False)

# Initialize accumulators
total_mae_plane, total_psnr_plane = 0, 0
total_mae_vortex, total_psnr_vortex = 0, 0
hist_errors_plane, hist_errors_vortex = [], []

print("Starting memory-safe evaluation")
with torch.no_grad():
    for (inputs_plane, labels), (inputs_vortex, _) in tqdm(zip(test_loader_plane, test_loader_vortex), total=len(test_loader_plane), desc="Evaluating Test Set"):
        inputs_plane, inputs_vortex, labels = inputs_plane.to(device), inputs_vortex.to(device), labels.to(device)
        
        preds_plane = model_plane(inputs_plane)
        total_mae_plane += nn.L1Loss()(preds_plane, labels).item() * labels.size(0)
        total_psnr_plane += psnr(labels, preds_plane) * labels.size(0)
        hist_errors_plane.append(torch.abs(preds_plane - labels).cpu().numpy().flatten())

        preds_vortex = model_vortex(inputs_vortex)
        total_mae_vortex += nn.L1Loss()(preds_vortex, labels).item() * labels.size(0)
        total_psnr_vortex += psnr(labels, preds_vortex) * labels.size(0)
        hist_errors_vortex.append(torch.abs(preds_vortex - labels).cpu().numpy().flatten())
        
        del inputs_plane, inputs_vortex, labels, preds_plane, preds_vortex
        gc.collect()

avg_mae_plane = total_mae_plane / NUM_TEST; avg_psnr_plane = total_psnr_plane / NUM_TEST
avg_mae_vortex = total_mae_vortex / NUM_TEST; avg_psnr_vortex = total_psnr_vortex / NUM_TEST
print("\nQuantitative Results on Test Set")
print(f"Plane Wave Model -> MAE: {avg_mae_plane:.6f} | PSNR: {avg_psnr_plane:.2f} dB")
print(f"Vortex Model     -> MAE: {avg_mae_vortex:.6f} | PSNR: {avg_psnr_vortex:.2f} dB")
error_plane_full = np.concatenate(hist_errors_plane)
error_vortex_full = np.concatenate(hist_errors_vortex)
np.save(os.path.join(PLOT_SAVE_PATH, 'error_data_for_hist.npy'), {'plane': error_plane_full, 'vortex': error_vortex_full})
print(f"\nSaved error data to {os.path.join(PLOT_SAVE_PATH, 'error_data_for_hist.npy')}")

# Cleanup
del hist_errors_plane, hist_errors_vortex, error_plane_full, error_vortex_full
gc.collect()
print("\nEvaluation complete. You can now run the final plotting cell.")

Starting memory-safe evaluation


Evaluating Test Set: 100%|█████████████████████████████████████████████████████████████| 25/25 [05:31<00:00, 13.28s/it]



Quantitative Results on Test Set
Plane Wave Model -> MAE: 0.206309 | PSNR: 12.03 dB
Vortex Model     -> MAE: 0.066045 | PSNR: 16.03 dB

Saved error data to ..\results\plots\error_data_for_hist.npy

Evaluation complete. You can now run the final plotting cell.


In [ ]:
print("\nGenerating Error Histogram Plot")

error_data_path = os.path.join(PLOT_SAVE_PATH, 'error_data_for_hist.npy')
if os.path.exists(error_data_path):
    error_data = np.load(error_data_path, allow_pickle=True).item()
    error_plane_full = error_data['plane']
    error_vortex_full = error_data['vortex']
    MAX_SAMPLES_FOR_HIST = 200000 
    if len(error_plane_full) > MAX_SAMPLES_FOR_HIST:
        print(f"Subsampling error data from {len(error_plane_full)} to {MAX_SAMPLES_FOR_HIST} points for plotting.")
        indices = np.random.choice(len(error_plane_full), MAX_SAMPLES_FOR_HIST, replace=False)
        error_plane_plot = error_plane_full[indices]
        error_vortex_plot = error_vortex_full[indices]
    else:
        error_plane_plot = error_plane_full
        error_vortex_plot = error_vortex_full

    plt.figure(figsize=(10, 6))
    plt.hist(error_plane_plot, bins=100, range=(0, 0.5), density=True, alpha=0.7, label='Plane Wave Error')
    plt.hist(error_vortex_plot, bins=100, range=(0, 0.5), density=True, alpha=0.7, label='Vortex Error')
    plt.title('Distribution of Absolute Pixel Errors on Test Set', fontsize=16)
    plt.xlabel('Absolute Error'); plt.ylabel('Probability Density'); plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(PLOT_SAVE_PATH, 'error_histograms.png'))
    plt.show()

    del error_data, error_plane_full, error_vortex_full, error_plane_plot, error_vortex_plot
    plt.close('all')
    gc.collect()
    print(f"Saved error histogram plot to {PLOT_SAVE_PATH}")

else:
    print("Error: Could not find 'error_data_for_hist.npy'. Please run the quantitative evaluation cell first.")



Generating Error Histogram Plot
Subsampling error data from 6553600 to 200000 points for plotting.
